# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## Opsætning


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Opgaveopdeling

Opgaveopdeling er kernen i planlægningsdesignmønsteret. I stedet for at bede en enkelt agent om
at håndtere en kompleks forespørgsel fra ende til anden, deler vi problemet op i mindre, veldefinerede **delopgaver**.
Hver delopgave tildeles en specialistagent (f.eks. fly, hoteller, aktiviteter) med klare
prioriteter og afhængighedsordning.

Denne tilgang giver flere fordele:
- **Klarhed**: hver delopgave har et enkelt ansvar.
- **Parallelisme**: uafhængige delopgaver kan køre samtidigt.
- **Pålidelighed**: fejl isoleres til individuelle delopgaver.
- **Budgetopfølgning**: omkostninger estimeres pr. delopgave og opsummeres.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Oprettelse af en planlægningsagent med struktureret output

Planlægningsagenten fungerer som en **receptionskoordinator**. Givet en overordnet rejseanmodning
genererer den en struktureret `TravelPlan` — opdeler anmodningen i delopgaver, sætter prioriteter,
og identificerer afhængigheder, så en concierge eller udførelseslag kan udføre arbejdet.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Udførelse af en Plan med Specialistværktøjer

Når receptionisten har udarbejdet en struktureret plan, udfører **concierge-agenten** den.
Hvert specialistværktøj håndterer én kategori af delopgave (fly, hoteller, aktiviteter). Concierge-agenten
gennemgår planens delopgaver i afhængighedsorden og sender hver enkelt videre til det
relevante værktøj.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Resumé

I denne lektion lærte du **Planlægningsdesignmønsteret** for AI-agenter:

1. **Opgavedekomponering** — En planlægningsagent i receptionen opdeler en kompleks rejseanmodning i
   strukturerede delopgaver ved hjælp af Pydantic-modeller, og tildeler hver enkelt til en specialistagent med prioriteter
   og afhængigheder.
2. **Struktureret output** — Ved at videregive et `response_format` returnerer agenten et valideret
   `TravelPlan`-objekt i stedet for frit tekstformat, hvilket gør efterfølgende behandling pålidelig.
3. **Planudførelse** — En concierge-agent gennemgår delopgaverne ved hjælp af specialistværktøjer
   (`book_flight`, `reserve_hotel`, `book_activity`) for at gennemføre planlægningen og rapportere resultater.

Dette mønster adskiller *hvad der skal gøres* (planlægning) fra *hvordan det gøres* (udførelse), hvilket gør agenter
mere modulære, testbare og lettere at udvide.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
